# Immunization DSS: data to trained forecasting models (Colab pipeline)

One cell per stage of `docs/12_DATA_TO_TRAINING_PIPELINE.md`. Every stage calls the same `immdss` command that is
versioned in the repository, so the notebook holds no model code of its own. All model fitting happens here, on
Colab, never on the development laptop (D-14).

**Before running:** Runtime > Change runtime type > GPU (T4 is enough). Then Runtime > Run all.

| Stage | Cell | Output (under `/content/immunization-dss/results/`) |
|---|---|---|
| 0 | Settings, environment, clone at a fixed commit, install, tests | `environment.txt` |
| 2, 3 | Generate the synthetic data with seed 42 and validate it; check it is byte-identical to the evidence run | `data/synthetic/<run_id>/` |
| 4 | EDA of the generated data | `eda/` |
| 7, 8 | Build and clean the weekly training series; series EDA | `series/`, `series_eda/` |
| 9, 10 | Rolling-origin backtest: baselines, then B1, B2, SARIMA and GRU | `backtest_baselines/`, `backtest/` |
| 11 | Compare models, per vaccine, select the best per series (D-21), bias against true demand | `backtest/` |
| 12 | Fit the selected models on all weeks, forecast the next 4 weeks | `backtest/final_forecasts.csv`, `backtest/gru.keras` |
| - | Package everything for download | `results_<run_id>_<commit>.zip` |

The data is fully synthetic (D-02). No file is uploaded: the data is regenerated here from the seed and checked
against the reference hashes in `analytics/configs/evidence_hashes.json`.

## 0. Settings

In [ ]:
REPO_URL = "https://github.com/shantelle04/immunization-dss.git"
REF = "phase/4-models"  # branch, tag or commit; use a tag (for example p2) for a result cited in the thesis
SEED = 42
MODELS = "b1,b2,sarima,gru"
WORKDIR = "/content/immunization-dss"

### Environment and GPU

In [ ]:
import sys, subprocess
print(sys.version)
assert sys.version_info >= (3, 12), "the analytics package needs Python 3.12 or later"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "No GPU: Runtime > Change runtime type > GPU")

### Clone the repository at a fixed commit and install the analytics package

In [ ]:
import os
if not os.path.exists(WORKDIR):
    !git clone --quiet {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git fetch --quiet origin {REF} && git checkout --quiet --detach FETCH_HEAD
HEADS = !git rev-parse HEAD FETCH_HEAD
assert len(HEADS) == 2 and HEADS[0] == HEADS[1], f"checkout of {REF} failed: {HEADS}"
COMMIT = HEADS[0][:7]
print("commit", COMMIT)
!pip install --quiet -e "analytics[dev,train]"
!mkdir -p results && pip freeze > results/environment.txt

### Unit tests of the pipeline code (series, cleaning, splits, metrics, baselines, selection)

In [ ]:
!python -m pytest -q analytics/tests

## 2 and 3. Generate and validate the synthetic data

`immdss simulate` rebuilds the evidence run from seed 42 (about 2 to 4 minutes) and runs the 11 validation checks.
The next cell compares every file hash with the reference. A mismatch means a library version changed the numbers:
the run is still valid but is a different run, and must be logged as such, not cited as the evidence run.

In [ ]:
!immdss simulate --seed {SEED}
RUN_ID = open("analytics/configs/evidence_run.txt").read().strip()
RUN_DIR = f"data/synthetic/{RUN_ID}"
print("run", RUN_ID)

In [ ]:
import json
reference = json.load(open("analytics/configs/evidence_hashes.json"))
manifest = json.load(open(f"{RUN_DIR}/manifest.json"))
assert manifest["run_id"] == reference["run_id"], "different settings: not the evidence run"
mismatched = [f for f, sha in reference["files"].items() if manifest["files"][f]["sha256"] != sha]
print("generator code identical:", manifest["code_sha256"] == reference["code_sha256"])
print("files identical to the evidence run:", len(reference["files"]) - len(mismatched), "of", len(reference["files"]))
assert not mismatched, f"differs from the evidence run: {mismatched}"
validation = json.load(open(f"{RUN_DIR}/validation_report.json"))
print("validation passed:", validation["passed"])

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{RUN_DIR}/validation_report.md").read()))

## 4. Exploratory data analysis of the generated data (figures F-D1 to F-D8, table T-5.1)

In [ ]:
!immdss eda {RUN_DIR} --out results/eda

In [ ]:
import glob
from IPython.display import Image
display(Markdown(open(f"results/eda/EDA_{RUN_ID}.md").read()))
for png in sorted(glob.glob("results/eda/F-D*.png")):
    print(png)
    display(Image(png, width=900))

## 7 and 8. Build and clean the weekly training series

Only issue transactions from `app/stock_transactions.csv` are used (what a real clinic system records); `truth/` is
not read. Rules T-01 (a week without issues is 0), T-02 (stock window only) and T-03 (stock-out weeks flagged from
the ledger balance).

In [ ]:
!immdss build-series {RUN_DIR} --out results/series
SERIES = f"results/series/{RUN_ID}/weekly_issues.csv"
series_manifest = json.load(open(f"results/series/{RUN_ID}/series_manifest.json"))
print("series identical to the reference:", series_manifest["sha256"] == reference["weekly_issues_sha256"])

### Series EDA: level, zeros, stock-outs, seasonality and autocorrelation per vaccine

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
os.makedirs("results/series_eda", exist_ok=True)
s = pd.read_csv(SERIES, parse_dates=["week_start"])
summary = s.groupby("antigen_code").agg(
    series=("facility_code", "nunique"),
    mean_weekly_doses=("issued", "mean"),
    share_zero_weeks=("issued", lambda x: (x == 0).mean()),
    share_stockout_weeks=("stockout_flag", "mean"),
).round(3)
acf = {}
for ant, g in s.groupby("antigen_code"):
    total = g.groupby("week_start")["issued"].sum()
    acf[ant] = {f"acf_lag{l}": round(total.autocorr(l), 3) for l in (1, 4, 52)}
summary = summary.join(pd.DataFrame(acf).T)
summary.to_csv("results/series_eda/T-M1_series_summary.csv")
display(summary)

total = s.pivot_table(index="week_start", columns="antigen_code", values="issued", aggfunc="sum")
ax = total.plot(figsize=(12, 5), colormap="tab10", linewidth=1)
ax.set(title="Weekly doses issued, all 12 facilities", xlabel="Week", ylabel="Doses")
plt.tight_layout(); plt.savefig("results/series_eda/F-M1_weekly_issues_by_vaccine.png", dpi=150); plt.show()

s["week_of_year"] = s["week_start"].dt.isocalendar().week.clip(upper=52)
seasonal = s.groupby(["week_of_year", "antigen_code"])["issued"].mean().unstack()
ax = (seasonal / seasonal.mean()).plot(figsize=(12, 5), colormap="tab10", linewidth=1)
ax.set(title="Seasonality: mean doses per week of year, relative to the vaccine's mean", xlabel="Week of year",
       ylabel="Relative level")
plt.tight_layout(); plt.savefig("results/series_eda/F-M2_seasonality.png", dpi=150); plt.show()

## 9 and 10. Rolling-origin backtest

Six origins of 4 weeks cover the last 24 weeks. At each origin a model is fitted only on the weeks before it and
forecasts the next 4 (no test week is ever in a training window; the code asserts it). First the two baselines as a
quick check, then every model, which is the run that counts.

In [ ]:
!immdss backtest {RUN_DIR} --series {SERIES} --models b1,b2 --seed {SEED} --out results/backtest_baselines

In [ ]:
%%time
!immdss backtest {RUN_DIR} --series {SERIES} --models {MODELS} --seed {SEED} --out results/backtest

## 11. Compare the models and select the best per series

MASE under 1 means better than the seasonal naive forecast (doc 05). Selection rule D-21: the GRU where a series has
at least 104 training weeks and the GRU beats B1; otherwise SARIMA; otherwise the better baseline. `truth_bias`
compares forecasts with the true demand from `truth/`, after forecasting, for evaluation only.

In [ ]:
B = "results/backtest"
overall = pd.read_csv(f"{B}/comparison_overall.csv"); display(overall)
by_vaccine = pd.read_csv(f"{B}/comparison_by_vaccine.csv"); display(by_vaccine)
selection = pd.read_csv(f"{B}/selection.csv")
display(selection.groupby(["model", "reason"]).size().rename("series").reset_index())
if os.path.exists(f"{B}/truth_bias.csv"):
    display(pd.read_csv(f"{B}/truth_bias.csv"))

metrics = pd.read_csv(f"{B}/metrics.csv")
order = overall["model"].tolist()
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([metrics.loc[metrics.model == m, "mase"].dropna() for m in order], showfliers=False)
ax.set_xticks(range(1, len(order) + 1), order)
ax.axhline(1, color="grey", linestyle="--", linewidth=1)
ax.set(title="MASE per series by model (below the line beats seasonal naive)", ylabel="MASE")
plt.tight_layout(); plt.savefig(f"{B}/F-M3_mase_by_model.png", dpi=150); plt.show()

### Stock-out alerts against ground truth (doc 05 section 2, D-38)

One alert decision per series at each backtest origin, using the selected model's forecast and ledger data before the origin; scored against true stock-out weeks in `truth/` (evaluation only). D-07 target: recall at least 0.95, precision reported.

In [ ]:
!immdss evaluate-alerts {RUN_DIR} --results results/backtest
alerts = json.load(open("results/backtest/alert_metrics.json"))
display(pd.DataFrame(alerts["sensitivity"]))
display(pd.DataFrame(alerts["by_vaccine"]).T)

### Alert improvement study (D-43): weekly decisions, delivery-history gate, risk score

The alert rule is decided every week over the last 24 weeks (the application re-checks stock every night), with the selected models' forecasts at weekly cut-offs. A late-delivery warning is gated by each series' own delivery history, and a logistic-regression risk score is trained on ledger features with the ledger's own stock-out flag as the label. The gate and the risk threshold are set on the 24 weeks before the test weeks; `truth/` is read only to score the test weeks. Definitions: doc 05 section 2.

In [ ]:
%%time
!immdss alert-study {RUN_DIR} --series {SERIES} --results results/backtest --seed {SEED} --out results/alert_study
display(pd.read_csv("results/alert_study/alert_study.csv"))

## 12. Fit the selected model per series on all weeks and forecast the next 4 weeks

In [ ]:
!immdss train-final --series {SERIES} --results results/backtest --seed {SEED}
display(pd.read_csv("results/backtest/final_forecasts.csv").head(12))

## Package the results

Download the zip (or copy it to Google Drive), unzip it and run `python3 scripts/dev.py forecasts --results <folder>/backtest` to use the trained models in the app (D-39). It holds the manifests, metrics, comparisons, selection, final
forecasts, the GRU file, the EDA figures and `environment.txt`. The training log entry
(`docs/logs/TRAINING_RUN_LOG.md`) is written from its `backtest/manifest.json`; no number enters a document
without it.

In [ ]:
import shutil
archive = shutil.make_archive(f"/content/results_{RUN_ID}_{COMMIT}", "zip", "results")
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass